# OCR 캐시 생성 (Colab) — PaddleOCR-VL-1.6 Spotting

split 전체를 한 번 OCR 해서 **재사용 가능한 캐시**를 만듭니다.
이후 zero-shot 실험은 이 JSONL만 읽으면 되고, OCR을 다시 돌리지 않습니다.

## ⚠ Colab에서 먼저 알아야 할 것

- **런타임 - 런타임 유형 변경 - GPU (T4 이상)** 로 먼저 바꾸세요
- train 6,714장은 T4 기준 **15~20시간**입니다. Colab 무료는 보통 12시간 전에 끊깁니다
- 그래서 **결과를 Google Drive에 주기적으로 저장**하고, 다시 실행하면 **이어서** 진행합니다
- 끊기면 이 노트북을 처음부터 다시 Run All 하면 됩니다. 이미 끝난 장은 건너뜁니다
- 먼저 `LIMIT = 20`으로 한 번 돌려 형식과 속도를 확인한 뒤 `None`으로 바꾸길 권합니다

## 셀 구성

| 셀 | 내용 | GPU |
|---|---|---|
| 1 | 패키지 설치 (이후 **세션 다시 시작** 필요) | 불필요 |
| 2 | Drive 마운트 · 데이터 압축 해제 | 불필요 |
| 3 | 라이브러리 import | 불필요 |
| 4 | **파라미터** | 불필요 |
| 5 | OCR 엔진 정의 | 불필요 |
| 6 | 데이터 로드·검증 | 불필요 |
| 7 | 이어하기 준비 (Drive → 로컬) | 불필요 |
| 8 | **OCR 실행** | **필요** |
| 9 | 캐시 점검 | 불필요 |

## 1. 패키지 설치

PaddleOCR-VL-1.6은 **transformers 5.x**가 필요합니다. Colab 기본은 4.x라 반드시 설치해야 합니다.

**설치 후 반드시 `런타임 - 세션 다시 시작`을 하고, 이 셀은 건너뛴 채 2번부터 다시 실행하세요.**

In [ ]:
!pip -q install "transformers>=5.3.0,<6" accelerate
!pip -q install pillow pandas tqdm

In [ ]:
import transformers, torch
print("transformers", transformers.__version__, "/ torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert transformers.__version__ >= "5", "세션을 다시 시작한 뒤 2번 셀부터 실행하세요."

## 2. Drive 마운트 · 데이터 준비

Drive에는 두 가지를 둡니다.
- **입력**: `ssafy-16-2-ai.zip` (대회 데이터)
- **출력**: OCR 캐시. 런타임이 끊겨도 살아남도록 Drive에 씁니다

압축 해제는 몇 분 걸립니다. 이미 `/content/data`가 있으면 건너뜁니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"

assert os.path.isfile(ZIP_PATH), f"zip을 찾지 못했습니다: {ZIP_PATH}"
if list(Path("/content").glob("**/train.csv")):
    print("데이터가 이미 있습니다. 압축 해제를 건너뜁니다.")
else:
    !7z x -y -p"$ZIP_PASSWORD" -o/content/ "$ZIP_PATH"

# 압축 구조에 따라 /content/data 가 아닐 수 있으므로 실제 위치를 확인합니다.
found = [str(c.parent) for c in Path("/content").rglob("train.csv") if "drive" not in c.parts]
print("train.csv 위치:", found if found else "찾지 못함")
!ls /content

## 3. 라이브러리

In [ ]:
import gc, json, shutil, time
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

Image.MAX_IMAGE_PIXELS = None      # 큰 이미지 경고 해제
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# T4는 bf16을 지원하지 않으므로 fp16으로 떨어집니다. 공식 예시는 bf16입니다.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device:", DEVICE, "/ dtype:", DTYPE)

## 4. 파라미터

여기만 바꾸면 됩니다.

- `SPLIT` — `train`은 정답이 모두 있어 zero-shot 성능 측정이 가능합니다. `test`에는 정답 컬럼이 없습니다
- `MAX_NEW_TOKENS` — 이미지 해상도가 아니라 **글자량**이 좌우합니다 (픽셀수와의 상관 r=0.19).
  로컬 실측: 1024에서 잘림 약 25%. 1024→2048로 올려도 커버율이 늘지 않아(반복 루프가 원인) 1024를 권합니다
- `PROMPT` — `Spotting:`(텍스트+좌표) / `OCR:`(텍스트만, 토큰 약 절반).
  그 외 문장은 모델이 무시하고 Spotting으로 처리합니다
- `SYNC_EVERY` — 이 장수마다 Drive로 복사합니다. 런타임이 끊겨도 여기까지는 보존됩니다

In [ ]:
SPLIT = "train"             # train / test / dev
MAX_NEW_TOKENS = 1024
PROMPT = "Spotting:"
LIMIT = None               # 전체 처리. 정수를 넣으면 앞 N장만 (시험용)
SYNC_EVERY = 100           # Drive 저장 주기(장)

DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")   # 데이터 zip과 산출물이 있는 폴더

MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.6"
# 모델 1.8GB. Colab의 기본 HF 캐시는 세션과 함께 사라지므로
# Drive에 캐시하면 런타임을 새로 띄워도 다시 받지 않습니다.
CACHE_MODEL_ON_DRIVE = True
MODEL_CACHE = (DRIVE_BASE / "hf_cache") if CACHE_MODEL_ON_DRIVE else None
if MODEL_CACHE:
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("/content/data")   # 6번 셀에서 실제 위치로 다시 잡습니다   # 6번 셀에서 실제 위치로 다시 잡습니다
LOCAL_CACHE = Path("/content/ocr_cache")                       # 빠른 로컬 디스크
DRIVE_CACHE = DRIVE_BASE / "ocr_cache"                         # 영속 저장소
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

FILENAME = f"{SPLIT}_spotting.jsonl"
LOCAL_PATH, DRIVE_PATH = LOCAL_CACHE / FILENAME, DRIVE_CACHE / FILENAME
print(f"{SPLIT} / {PROMPT!r} / max_new_tokens={MAX_NEW_TOKENS} / limit={LIMIT}")
print("영속 저장:", DRIVE_PATH)
print("모델 캐시:", MODEL_CACHE or "세션 임시(재시작 시 1.8GB 재다운로드)")

## 5. OCR 엔진

로컬 `ocr_engine.py`와 같은 내용을 Colab용으로 인라인한 것입니다 (파일 의존성 없이 단독 실행되도록).

공식 근거: [모델 카드](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6)
- `task="spotting"` → 프롬프트 `Spotting:`
- 두 변이 모두 1500 미만이면 2배 확대(LANCZOS)
- `max_pixels = 2048*28*28` (spotting 전용 값)

In [ ]:
OCR_MAX_PIXELS = 2048 * 28 * 28      # 1,605,632 = 비전 토큰 2048개분
SPOTTING_UPSCALE_THRESHOLD = 1500

def spotting_image(image_path):
    """공식 spotting 전처리. 공통 축소 규칙을 적용하지 않습니다."""
    with Image.open(image_path) as im:
        im = im.convert("RGB")
    if im.width < SPOTTING_UPSCALE_THRESHOLD and im.height < SPOTTING_UPSCALE_THRESHOLD:
        im = im.resize((im.width * 2, im.height * 2), Image.Resampling.LANCZOS)
    return im

def _read_pixel_bounds(ip):
    """transformers 5.x 내장 클래스는 size가 SizeDict(shortest_edge/longest_edge = 픽셀 수),
    모델 폴더의 remote code 클래스는 min_pixels/max_pixels 속성을 씁니다."""
    size = getattr(ip, "size", None)
    def pick(*keys):
        for key in keys:
            v = size.get(key) if isinstance(size, dict) else getattr(size, key, None)
            if v:
                return int(v)
        return None
    low = pick("shortest_edge", "min_pixels") or getattr(ip, "min_pixels", None)
    high = pick("longest_edge", "max_pixels") or getattr(ip, "max_pixels", None)
    if not low or not high:
        raise ValueError(f"OCR 픽셀 설정을 찾지 못했습니다: {type(ip).__name__}, size={size}")
    return int(low), int(high)

def _probe_pixels(proc, edge=2048):
    grid = proc.image_processor(images=Image.new("RGB", (edge, edge)),
                               return_tensors="pt")["image_grid_thw"]
    return int(grid[0][1]) * int(grid[0][2]) * proc.image_processor.patch_size ** 2

def configure_ocr_pixels(proc, max_pixels=OCR_MAX_PIXELS):
    """픽셀 상한을 spotting 값으로 올리고 실측으로 검증합니다.

    apply_chat_template(images_kwargs=...)는 transformers 5.x에서 무시되므로
    프로세서 객체에 직접 설정합니다.
    """
    ip = proc.image_processor
    min_pixels, default_max = _read_pixel_bounds(ip)
    before = _probe_pixels(proc)
    size = getattr(ip, "size", None)
    if isinstance(size, dict):
        upd = dict(size)
        keys = ("shortest_edge", "longest_edge") if size.get("shortest_edge") else ("min_pixels", "max_pixels")
        upd[keys[0]], upd[keys[1]] = min_pixels, max_pixels
        ip.size = upd
    elif size is not None and getattr(size, "shortest_edge", None):
        size.shortest_edge, size.longest_edge = min_pixels, max_pixels
    if hasattr(ip, "min_pixels"):
        ip.min_pixels = min_pixels
    if hasattr(ip, "max_pixels"):
        ip.max_pixels = max_pixels
    after = _probe_pixels(proc)
    if not before < after <= max_pixels:
        raise RuntimeError(f"OCR 픽셀 상한 미적용: 2048px 입력이 {before} -> {after} "
                           f"(기대: {default_max} 초과, {max_pixels} 이하)")
    print(f"OCR 픽셀 범위 {min_pixels}~{max_pixels} 적용 확인 "
          f"(2048px 입력: {before} -> {after} 픽셀)")

def load_ocr():
    kw = {"cache_dir": str(MODEL_CACHE)} if MODEL_CACHE else {}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **kw)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=DTYPE, **kw).to(DEVICE).eval()
    configure_ocr_pixels(proc)
    return model, proc

@torch.inference_mode()
def run_spotting(model, proc, image, max_new_tokens, prompt=PROMPT):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = proc.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    prefix_len = inputs["input_ids"].shape[-1]
    # 이 모델의 generation_config는 use_cache=False라서 매 스텝 전체를 다시 계산합니다.
    # greedy이므로 캐시를 켜도 생성 토큰은 동일하고 속도만 회복됩니다(로컬 실측 25배 차이).
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, use_cache=True)
    new = out[0, prefix_len:]
    eos = model.generation_config.eos_token_id
    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))
    truncated = len(new) >= max_new_tokens and (not len(new) or int(new[-1]) not in eos_ids)
    return proc.decode(new, skip_special_tokens=True).strip(), bool(truncated), int(len(new))

print("OCR 엔진 준비 완료")

## 6. 데이터 로드·검증
이미지가 실제로 존재하는지 먼저 확인합니다. 긴 작업 도중에 파일 누락을 발견하는 상황을 막습니다.

In [ ]:
def find_data_dir(split):
    """압축 해제 위치가 zip 구조에 따라 달라지므로 실제 위치를 찾습니다."""
    for base in (Path("/content/data"), Path("/content")):
        if (base / f"{split}.csv").is_file() and (base / split).is_dir():
            return base
    for csv in Path("/content").rglob(f"{split}.csv"):
        if "drive" in csv.parts:          # Drive 원본은 제외
            continue
        if (csv.parent / split).is_dir():
            return csv.parent
    listing = sorted(x.name for x in Path("/content").iterdir())
    raise FileNotFoundError(
        f"{split}.csv 와 {split}/ 폴더를 /content 아래에서 찾지 못했습니다. "
        f"/content 내용: {listing}. "
        "2번 셀의 압축 해제가 성공했는지, ZIP_PATH와 비밀번호가 맞는지 확인하세요.")

DATA_DIR = find_data_dir(SPLIT)
print("데이터 위치:", DATA_DIR)

df = pd.read_csv(DATA_DIR / f"{SPLIT}.csv", keep_default_na=False)
assert df.id.is_unique and df.path.is_unique, "id 또는 path 중복"
missing = [p for p in df.path if not (DATA_DIR / p).is_file()]
assert not missing, f"이미지 누락 {len(missing)}개. 예: {missing[:3]}"
if LIMIT:
    df = df.head(LIMIT).copy()
print(f"{len(df)}장 확인 완료")

## 7. 이어하기 준비

Drive에 있던 캐시를 로컬로 가져와 이어서 씁니다.
매 줄을 Drive에 직접 쓰면 느리고 불안정하므로, **로컬에 쓰고 주기적으로 Drive에 복사**합니다.

설정이 다른 결과가 한 파일에 섞이면 이후 실험이 조용히 오염되므로,
`max_new_tokens`가 다르면 중단합니다.

In [ ]:
if DRIVE_PATH.is_file() and not LOCAL_PATH.is_file():
    shutil.copy(DRIVE_PATH, LOCAL_PATH)
    print(f"Drive에서 이어받기: {DRIVE_PATH}")

def load_done(path, max_new_tokens):
    if not path.is_file():
        return set()
    done, kept, caps = set(), [], set()
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            done.add(item["id"]); caps.add(item.get("max_new_tokens")); kept.append(line)
        except (json.JSONDecodeError, KeyError):
            print("경고: 손상된 줄 1개를 버립니다 (끊긴 마지막 줄로 추정)")
    if len(kept) != len(done):
        path.write_text("\n".join(kept) + "\n", encoding="utf-8")
    other = {c for c in caps if c is not None and c != max_new_tokens}
    if other:
        raise SystemExit(f"기존 캐시는 max_new_tokens={sorted(other)}로 생성되었습니다. "
                         f"지금 설정은 {max_new_tokens}입니다. 같은 값으로 이어하거나 "
                         f"{path} 와 {DRIVE_PATH} 를 지우고 다시 만드세요.")
    return done

done = load_done(LOCAL_PATH, MAX_NEW_TOKENS)
todo = df[~df.id.isin(done)]
print(f"완료 {len(done)}장 / 남음 {len(todo)}장")

(DRIVE_CACHE / f"{SPLIT}_spotting.meta.json").write_text(json.dumps({
    "split": SPLIT, "rows": len(df), "max_new_tokens": MAX_NEW_TOKENS, "prompt": PROMPT,
    "model": MODEL_ID, "max_pixels": OCR_MAX_PIXELS,
    "upscale_threshold": SPOTTING_UPSCALE_THRESHOLD, "dtype": str(DTYPE),
}, ensure_ascii=False, indent=2), encoding="utf-8")

## 8. OCR 실행

모델을 처음 받을 때 2GB 정도 다운로드가 있습니다.

진행 중 `SYNC_EVERY`장마다 Drive에 저장됩니다. 런타임이 끊기면 마지막 저장 시점까지 남고,
이 노트북을 다시 Run All 하면 그 지점부터 이어서 진행합니다.

In [ ]:
def sync_to_drive():
    if LOCAL_PATH.is_file():
        shutil.copy(LOCAL_PATH, DRIVE_PATH)

if todo.empty:
    print("이미 모두 완료되었습니다.")
else:
    model, proc = load_ocr()
    torch.cuda.reset_peak_memory_stats()
    truncated_count, elapsed_total, n_new = 0, 0.0, 0
    try:
        with LOCAL_PATH.open("a", encoding="utf-8") as log:
            for row in tqdm(todo.to_dict("records"), desc=f"OCR {SPLIT}"):
                torch.cuda.synchronize(); start = time.perf_counter()
                im = spotting_image(DATA_DIR / row["path"])
                text, capped, n_tokens = run_spotting(model, proc, im, MAX_NEW_TOKENS)
                torch.cuda.synchronize(); seconds = time.perf_counter() - start
                elapsed_total += seconds; truncated_count += capped; n_new += 1
                log.write(json.dumps({
                    "id": row["id"], "path": row["path"], "ocr_raw": text,
                    "ocr_tokens": n_tokens, "ocr_truncated": capped,
                    "max_new_tokens": MAX_NEW_TOKENS, "prompt": PROMPT,
                    "seconds": seconds, "image_width": im.width, "image_height": im.height,
                }, ensure_ascii=False) + "\n")
                log.flush()
                if n_new % SYNC_EVERY == 0:
                    sync_to_drive()
                del im
    finally:
        sync_to_drive()        # 오류로 중단돼도 진행분은 Drive에 남깁니다
        del model, proc
        gc.collect(); torch.cuda.empty_cache()
    print(f"\n완료 {n_new}장 / 평균 {elapsed_total / max(n_new,1):.2f}초 "
          f"/ 총 {elapsed_total / 3600:.2f}시간")
    print(f"길이 제한 도달: {truncated_count}장 ({truncated_count / max(n_new,1):.1%})")
    print(f"GPU peak {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f} GiB")
    print("Drive 저장:", DRIVE_PATH)

## 9. 캐시 점검
GPU가 필요 없습니다. 3·4·6번 셀만 실행한 뒤 이 셀로 바로 와도 됩니다.

In [ ]:
cache = pd.read_json(LOCAL_PATH if LOCAL_PATH.is_file() else DRIVE_PATH, lines=True)
print(f"캐시 {len(cache)}행 / 대상 {len(df)}행")
assert cache.id.is_unique, "캐시에 id 중복"
missing_ids = set(df.id) - set(cache.id)
print("누락:", len(missing_ids), "" if not missing_ids else list(missing_ids)[:3])

# 컬럼명 empty는 DataFrame.empty 속성과 충돌하므로 is_empty를 씁니다.
cache["is_empty"] = cache.ocr_raw.astype(str).str.strip() == ""
print(f"\n빈 결과       {cache['is_empty'].sum():5d}장 ({cache['is_empty'].mean():.1%})")
print(f"길이 제한 도달 {cache['ocr_truncated'].sum():5d}장 ({cache['ocr_truncated'].mean():.1%})")
print(f"\n생성 토큰 분포:\n{cache.ocr_tokens.describe().round(1).to_string()}")
print(f"\n장당 평균 {cache.seconds.mean():.2f}초")
if len(cache) < len(df):
    remain = (len(df) - len(cache)) * cache.seconds.mean() / 3600
    print(f"→ 남은 {len(df) - len(cache)}장 예상: 약 {remain:.1f}시간")
print("\n샘플 2건:")
for _, r in cache.head(2).iterrows():
    print(f"  {r.id} ({r.ocr_tokens}토큰): {str(r.ocr_raw)[:200]}")

## 이후 실험에서 읽는 법

```python
import pandas as pd
cache = pd.read_json("/content/drive/MyDrive/SSAFY/AI_Challenge2/ocr_cache/train_spotting.jsonl", lines=True)
evidence = dict(zip(cache.id, cache.ocr_raw))     # id -> OCR 원문

train = pd.read_csv("/content/data/train.csv", keep_default_na=False)
train["ocr"] = train.id.map(evidence)
```

**저장 형식(JSONL)** — 한 줄 = JSON 하나 = 이미지 한 장:

| 필드 | 내용 |
|---|---|
| `id`, `path` | 이미지 식별자 |
| `ocr_raw` | OCR 원문. Spotting이면 글자마다 `<\|LOC_n\|>` 좌표 토큰 8개가 붙습니다 |
| `ocr_tokens` | 생성 토큰 수 |
| `ocr_truncated` | 길이 제한 도달 여부 |
| `max_new_tokens`, `prompt` | 이 줄을 만든 설정 |
| `seconds` | 소요 시간 |
| `image_width/height` | 전처리 후 크기 |

`ocr_raw`에 원문을 그대로 두므로 좌표 제거나 재포맷 같은 후처리는 나중에 얼마든지 가능합니다.

```python
import re
plain = re.sub(r"<\|LOC_\d+\|>", "", ocr_raw)   # 좌표 제거
```